# Observe With Tracing

## Implement MLflow Tracing for observability and debugging

MLflow Tracing provides comprehensive observability for your GenAI applications, allowing you to:

- **Debug** issues by inspecting inputs, outputs, and intermediate steps
- **Collect** user feedback to improve your models
- **Optimize** latency by identifying bottlenecks

Implementing MLflow Tracing only requires _a single line of code_. You can attach end user feedback from your UI to the corresponding trace so you can quickly debug user-reported issues.

![tracing-demo](https://i.imgur.com/tNYUHdC.gif)

## Install packages (only required if running in a Databricks Notebook)

In [ ]:
%pip install -U -r ../requirements.txt
dbutils.library.restartPython()

## Environment Setup

Load environment variables that connect to the MLflow server for trace logging and configure the banking agent to access Databricks resources.

In [ ]:
import sys
sys.path.append('../')
sys.path.append('../../')

import os
import mlflow
from mlflow_demo.utils import *

# Configure for the current participant: derives the object prefix, prompt name,
# model service, and experiment from YOUR username. The widgets only carry the
# shared catalog/schema/warehouse.
setup_databricks_notebook_env()

print('=== Environment Setup (derived from your username) ===')
print(f"prefix:        {os.environ['UC_TOOL_PREFIX']}")
print(f"catalog:       {os.environ['UC_CATALOG']}.{os.environ['UC_SCHEMA']}")
print(f"prompt:        {os.environ['PROMPT_NAME']}")
print(f"model service: {os.environ['LLM_MODEL_SERVICE']}")
print(f"experiment:    {os.environ['MLFLOW_EXPERIMENT_NAME']} (ID {os.environ['MLFLOW_EXPERIMENT_ID']})")
print(f"warehouse:     {os.environ['MLFLOW_TRACING_SQL_WAREHOUSE_ID']}")

import logging
logging.getLogger("urllib3").setLevel(logging.ERROR)
logging.getLogger("mlflow").setLevel(logging.ERROR)

# Step 1: Instrument with Automatic and Manual Tracing

MLflow provides comprehensive observability for your GenAI applications with **just a single line of code**. Automatic tracing works out-of-the-box with 20+ popular frameworks including OpenAI, LangChain, Anthropic, and more. Manual tracing captures your application's specific logic with custom spans, attributes, and detailed step-by-step tracking.

**Code References**

- See `ToolCallingAgent` class definition: `mlflow_demo/agent/agent.py`
- Automatic tracing setup:
  - `mlflow.openai.autolog()` at `mlflow_demo/agent/agent.py:712`
- Manual tracing with `@mlflow.trace`:
  - `execute_tool` at `mlflow_demo/agent/agent.py:384`
  - `predict` at `mlflow_demo/agent/agent.py:457`

**Documentation**

- **Automatic Tracing** - One-line setup for popular SDKs
- **Manual Tracing** - Custom spans and attributes

**Run the next cell to import the instrumented banking agent!**

In [ ]:
# Import the banking agent and MLflow types
# Note: This import must come after loading environment variables
from mlflow_demo.agent.agent import AGENT
from mlflow.types.responses import Message, ResponsesAgentRequest

print('Banking agent imported successfully')

# Step 2: Run the Banking Agent

Ask the agent banking questions using the `AGENT.predict()` API. This creates traces that you can view in the MLflow UI and attach feedback to later.

**Run the next cells to ask your first traced banking question!**

In [ ]:
# Define banking questions to test the agent
banking_questions = [
    "What's the balance on my checking account?",
    "Can you show my recent transactions from the last 30 days?",
    "What loans do I currently have?"
]

question = banking_questions[0]
print(f'Question: {question}')

In [ ]:
# Ask the agent a banking question - this will create MLflow traces!
request = ResponsesAgentRequest(
    input=[Message(role='user', content=question)]
)
response = AGENT.predict(request)

# Get the trace ID for this interaction
trace_id = mlflow.get_last_active_trace_id()

print(f'Trace ID: {trace_id}')

# Extract the assistant response text
for item in response.output:
    if hasattr(item, 'content'):
        for part in item.content:
            if hasattr(part, 'text'):
                print(f'Response: {part.text[:300]}...')

# Store the trace_id for use in feedback logging
current_trace_id = trace_id
print(f'Stored trace_id for feedback: {current_trace_id}')

# Generate MLflow UI links for this trace
print('\n' + '=' * 60)
generate_trace_links(current_trace_id)
print('=' * 60)

# Step 3: Log User Feedback to Traces

Attach user feedback to the traces you just created. This links customer satisfaction directly to specific agent responses, making it easy to debug issues and improve your prompts.

**Code References**

- `mlflow.log_feedback()` is called directly with the trace ID
- In production, the frontend passes the trace_id back when a customer submits feedback

**Documentation**

- [**Collecting User Feedback**](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/collect-user-feedback/) - Link feedback to traces

**Run the next cells to submit both positive and negative feedback examples!**

In [ ]:
# Example 1: Submit positive feedback
from mlflow.entities import AssessmentSource

if current_trace_id:
    mlflow.log_feedback(
        trace_id=current_trace_id,
        name='user_feedback',
        value=True,
        rationale='Great analysis of my account balances!',
        source=AssessmentSource(
            source_type='HUMAN',
            source_id='customer@bank.example.com'
        )
    )
    print('Positive feedback submitted:')
    print(f'   Trace ID: {current_trace_id}')

    # Generate MLflow UI links to view the trace with feedback
    print('\n' + '=' * 60)
    print('View the trace with your feedback attached:')
    generate_trace_links(current_trace_id)
    print('=' * 60)
else:
    print('No trace_id available - run the agent cell first')

In [ ]:
# Generate another trace with a different question for negative feedback
question2 = banking_questions[1]
print(f'Question: {question2}')

request2 = ResponsesAgentRequest(
    input=[Message(role='user', content=question2)]
)
response2 = AGENT.predict(request2)
second_trace_id = mlflow.get_last_active_trace_id()

print(f'Trace ID: {second_trace_id}')

# Submit negative feedback
if second_trace_id:
    mlflow.log_feedback(
        trace_id=second_trace_id,
        name='user_feedback',
        value=False,
        rationale="The response was missing some recent transactions. Could be more complete.",
        source=AssessmentSource(
            source_type='HUMAN',
            source_id='customer@bank.example.com'
        )
    )
    print('Negative feedback submitted:')
    print(f'   Trace ID: {second_trace_id}')

    # Generate MLflow UI links
    print('\n' + '=' * 60)
    print('View the trace with negative feedback attached:')
    generate_trace_links(second_trace_id)
    print('=' * 60)

# Step 4: Generate the workshop sample dataset

The notebooks that follow (quality metrics, find & fix, human review) all operate on a standard set of traced banking questions, tagged `sample_data=yes`. Run your agent over the 10 workshop questions below and watch the traces stack up in the MLflow UI — each one is fully traced (LLM spans, tool calls, outputs).

> On first import (Step 1) the agent also **registered your `<prefix>_bank_support_prompt`** in the UC Prompt Registry — that's the prompt notebook 6 will optimize later.

In [ ]:
# The 10 standardized workshop banking questions (kept in sync with
# workshop/banking_questions.jsonl). Sequential on purpose: get_last_active_trace_id()
# is process-global, so no thread pool here.
SAMPLE_QUESTIONS = [
    "What's the balance on my checking account?",
    'Can you show my recent transactions from the last 30 days?',
    'What loans do I currently have and what are the outstanding balances?',
    'How much available credit do I have on my credit card?',
    'What savings products do you offer with the best interest rates?',
    'Where is the nearest bank branch in Springfield?',
    'I see a transaction I don\'t recognize, how can I dispute it?',
    "What's the difference between my current account types?",
    'Can you summarize my banking relationship with the bank?',
    'What are the eligibility requirements for a mortgage loan?',
]

made, errors = 0, 0
for i, question in enumerate(SAMPLE_QUESTIONS, 1):
    try:
        print(f'[{i}/{len(SAMPLE_QUESTIONS)}] {question[:60]}...')
        AGENT.predict(ResponsesAgentRequest(input=[Message(role='user', content=question)]))
        trace_id = mlflow.get_last_active_trace_id()
        if trace_id:
            mlflow.set_trace_tag(trace_id=trace_id, key='sample_data', value='yes')
            made += 1
    except Exception as e:
        errors += 1
        print(f'  ERROR: {e}')

total = len(mlflow.search_traces(filter_string='tags.sample_data = "yes"', max_results=100, return_type='list'))
print(f'\nDone: {made} new traces tagged, {errors} errors — {total} sample_data traces in your experiment.')
generate_trace_links()

# Summary and Next Steps

Congratulations! You've successfully implemented MLflow tracing for the Banking Customer Support Agent.

## What You've Accomplished

- **Enabled automatic tracing** with `mlflow.openai.autolog()`
- **Generated traced agent responses** using `AGENT.predict()` with `@mlflow.trace`
- **Attached user feedback** to traces with `mlflow.log_feedback()`
- **Created custom spans** with detailed attributes and metrics

**Next steps**

- [Follow the quickstart to instrument your own app](https://docs.databricks.com/aws/en/mlflow3/genai/getting-started/tracing/)
- [Read the tracing documentation](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/)

## Exercise

Ask the agent about your own banking scenario. Which UC functions were called?

Use `mlflow.search_traces()` to find your trace and inspect the tool call spans. Look for spans with `span_type='TOOL'` to identify which UC functions the agent invoked.

## Exercise

Find a trace where the agent made multiple tool calls. What was the sequence?

Hint: Look at the trace's span hierarchy. Each `execute_tool` call is a child span of the `predict` span. The order of child spans tells you the tool call sequence.

## Exercise

Submit feedback on a trace where the response was inaccurate.

Ask the agent a question where you know the answer, then use `mlflow.log_feedback()` with `value=False` and a rationale explaining what was wrong.